In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

import logfire
logfire.configure()
logfire.instrument_pydantic_ai()

import nest_asyncio
nest_asyncio.apply()

Logfire project URL: https://logfire-eu.pydantic.dev/kang-mx/starter-project

In [2]:
from agent import faq_agent, SearchDeps
from ingest import build_index, load_faq_data

documents = load_faq_data()
index = build_index(documents)
deps = SearchDeps(index=index)

In [3]:
result = await faq_agent.run("How do I run Ollama locally?", deps=deps)
print(result.output)

15:48:21.053 faq_agent run
15:48:21.058   chat gemini-2.5-flash
15:48:22.616   running tool: search
15:48:22.645   chat gemini-2.5-flash
To run Ollama locally, follow these steps:

1.  **Install Ollama:**
    *   Visit [https://ollama.com/download](https://ollama.com/download) and download the installer for your operating system (macOS, Windows, or Linux).
    *   For macOS, download and install the `.pkg` file.
    *   For Windows, download and install the `.msi` file.
    *   For Linux, run the following command in your terminal:
        ```bash
        curl -fsSL https://ollama.com/install.sh | sh
        ```

2.  **Start a model:**
    *   Open a terminal and type:
        ```bash
        ollama run llama3
        ```
        This command will download the LLaMA 3 model (approximately 4GB), start it locally, and open a chat-like interface.

3.  **Test the Ollama local server:**
    *   Run the following command in your terminal:
        ```bash
        curl http://localhost:11434
 

In [9]:
import os
import requests
import dlt

READ_TOKEN = os.getenv("LOGFIRE_READ_TOKEN")
BASE_URL = "https://logfire-eu.pydantic.dev"

@dlt.resource(name="spans", write_disposition="replace")
def logfire_spans():
    query = """
    SELECT *
    FROM records
    WHERE otel_scope_name = 'pydantic-ai'
    ORDER BY start_timestamp DESC
    """
    headers = {"Authorization": f"Bearer {READ_TOKEN}"}
    response = requests.get(f"{BASE_URL}/v1/query", params={"sql": query}, headers=headers)
    response.raise_for_status()
    data = response.json()

    cols = data["columns"]

    if isinstance(cols, dict):
        # column-name -> list-of-values
        names = list(cols.keys())
        values = [cols[n] for n in names]
    else:
        # list of {"name":..., "values":...} objects
        names = [c["name"] for c in cols]
        values = [c["values"] for c in cols]

    n_rows = len(values[0]) if values else 0
    for i in range(n_rows):
        yield {names[j]: values[j][i] for j in range(len(names))}

pipeline = dlt.pipeline(
    pipeline_name="logfire_traces",
    destination="duckdb",
    dataset_name="agent_traces",
)

load_info = pipeline.run(logfire_spans())
print(load_info)

2026-07-19 16:01:31,502|[WARNING]|91275|128365902014272|dlt|validate.py|verify_normalized_table:113|In schema `logfire_traces`: The following columns in table 'spans' did not receive any data during this load and therefore could not have their types inferred:
  - attributes__model_request_parameters__output_object
  - attributes__model_request_parameters__prompted_output_template
  - attributes__model_request_parameters__thinking
  - deployment_environment
  - http_method
  - http_response_status_code
  - http_route
  - log_body
  - url_full
  - url_path
  - url_query

Unless type hints are provided, these columns will not be materialized in the destination.
One way to provide type hints is to use the 'columns' argument in the '@dlt.resource' decorator.  For example:

@dlt.resource(columns={'attributes__model_request_parameters__output_object': {'data_type': 'text'}})

2026-07-19 16:01:31,503|[WARNING]|91275|128365902014272|dlt|validate.py|verify_normalized_table:113|In schema `logfire

Pipeline logfire_traces load step completed in 1.07 seconds
1 load package(s) were loaded to destination duckdb and into dataset agent_traces
The duckdb destination used duckdb:////workspaces/llm-zoomcamp/dlt/logfire_traces.duckdb location to store data
Load package 1784476890.77677 is LOADED and contains no failed jobs


In [10]:
import duckdb
con = duckdb.connect("logfire_traces.duckdb")
con.sql("""
    SELECT COUNT(*) FROM information_schema.tables
    WHERE table_schema = 'agent_traces'
""").show()

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│           25 │
└──────────────┘



In [11]:
con.sql("SELECT table_name FROM information_schema.tables WHERE table_schema='agent_traces'").show()
con.sql("SELECT * FROM agent_traces.spans LIMIT 3").show()

┌───────────────────────────────────────────────────────────────────────────────────────────────┐
│                                          table_name                                           │
│                                            varchar                                            │
├───────────────────────────────────────────────────────────────────────────────────────────────┤
│ spans                                                                                         │
│ spans__attributes__gen_ai_input_messages                                                      │
│ spans__attributes__gen_ai_input_messages__parts                                               │
│ spans__attributes__gen_ai_input_messages__parts__result                                       │
│ spans__attributes__gen_ai_output_messages                                                     │
│ spans__attributes__gen_ai_output_messages__parts                                              │
│ spans__attributes_

In [12]:
con.sql("""
    SELECT
        trace_id,
        SUM(attributes__gen_ai_usage_input_tokens) AS total_input_tokens,
        COUNT(*) AS n_spans_with_tokens
    FROM agent_traces.spans
    WHERE trace_id = '019f7b1069bd708e7fdb8df63c194177'
      AND attributes__gen_ai_usage_input_tokens IS NOT NULL
    GROUP BY trace_id
""").show()

┌──────────────────────────────────┬────────────────────┬─────────────────────┐
│             trace_id             │ total_input_tokens │ n_spans_with_tokens │
│             varchar              │       int128       │        int64        │
├──────────────────────────────────┼────────────────────┼─────────────────────┤
│ 019f7b1069bd708e7fdb8df63c194177 │               1968 │                   2 │
└──────────────────────────────────┴────────────────────┴─────────────────────┘

